## Topic 1: Why Chunking Exists

### Concept, Intuition, Why It Exists

- Two separate, unrelated problems both get solved by the same fix — splitting documents into smaller pieces.
- **Problem 1 — context window limits**: an LLM can only accept a fixed amount of text in one call. A 20-page policy PDF ingested as one giant Document (from the JSON loader) cannot be stuffed whole into a prompt alongside a user's question — it simply won't fit, and even where it technically fits, stuffing it in wastes most of the available space on irrelevant pages.
- **Problem 2 — retrieval precision**: even ignoring context limits entirely, retrieval itself gets worse with large units. If "one Document" = one entire 20-page PDF, a semantic search for "premature withdrawal penalty" returns the *whole* PDF as the best match — including 19 pages that have nothing to do with the question. The embedding for a giant block of mixed-topic text is a blurry average of everything in it, matching nothing precisely.
- Chunking is the fix for both at once: split each Document's `page_content` into smaller, focused pieces *before* embedding, so each piece (a) fits comfortably in a prompt, and (b) embeds to a vector that actually represents one coherent idea, not a blur of twenty.
- Important distinction from earlier loader work: text files, CSV/Excel, and JSON loaders already made *some* granularity decisions at the loader level — text files as whole files, CSV/Excel as one row, JSON as one page. Chunking is a **second**, independent splitting step that can still apply *within* any of those units when the unit itself is still too large or covers more than one topic (e.g. a single JSON page containing three unrelated policy clauses).

### Internal Working

- Conceptually, chunking sits immediately after ingestion and immediately before embedding: Document → chunker → smaller Documents → embedder → vector store.
- Each chunk still needs to carry the same `page_content` + `metadata` shape from the Document pattern — a chunker doesn't replace the Document abstraction, it just produces *more, smaller* Documents from each input Document. Metadata typically gets extended with a chunk index so a retrieved chunk can still be traced back to its position within the original page/row.
- The core tension every chunking strategy has to navigate: chunks too large drift back into Problem 2 (blurry embeddings, context waste); chunks too small lose coherence (a chunk that's half a sentence embeds to something meaningless, and retrieval returns fragments a human — or the LLM — can't actually use).

### How It's Implemented In This Project

- This project already has a working `chunk_text()` function that performs **sentence-aware** chunking — it doesn't split mid-sentence, which is already better than the simplest possible approach (fixed character count, ignoring word/sentence boundaries).
- Current gap, named explicitly here so it's not silently inherited going forward: `chunk_text()` currently has **zero overlap** between consecutive chunks. The next topic covers exactly why that's a real gap and how to fix it.

### Real-World Issues, Edge Cases, Debugging

- **Silent context overflow**: if chunking is skipped or misconfigured and a Document is too large, some embedding APIs truncate silently rather than erroring — the back half of a policy page simply never gets embedded, and nobody notices until a question about that exact content returns nothing relevant.
- **Cost compounds with chunk count**: more, smaller chunks mean more embedding calls and more vector store entries — chunking too aggressively isn't free either, it's a trade against Problem 2's precision, not a free win.
- **Debugging retrieval misses often starts here**: a confidently wrong or "no relevant info found" answer is very often a chunking problem, not a model problem — sample the actual chunks a query's top results returned and check whether they're coherent, complete thoughts before assuming the retriever or the LLM is at fault.

### Design Decisions & Trade-offs

- Chunking is treated as a separate stage from ingestion, not folded into it, mirroring the same separation-of-concerns reasoning used for loaders — ingestion's job is "normalize sources into Documents," chunking's job is "split Documents into retrieval-sized pieces." Keeping them separate means the chunking strategy can be changed later without touching a single loader.
- Sentence-aware chunking over naive fixed-character chunking is itself a deliberate trade: slightly more complex to implement, in exchange for chunks that are actually coherent units of meaning instead of arbitrary character-count cuts that can land mid-word or mid-sentence.

### Alternatives & When To Use Each

- Don't chunk at all, embed whole Documents — acceptable only when every Document is already small and single-topic (this is effectively what the text/CSV loaders already achieve at the loader level, which is *why* they deliberately chose small, single-topic units in the first place).
- Chunk every Document uniformly regardless of source — simplest to implement, but ignores that different source types already arrive at different natural granularities, so applies the same logic everywhere even where it isn't needed (the same point made earlier about not chunking the keyword reference files).
- Chunk selectively, only where a Document is still too large or multi-topic after ingestion — more deliberate, requires checking Document size/content before deciding, which is the realistic production approach.

### Common Mistakes & Production Failures

- Treating chunking as a one-time decision made early and never revisited, even as document types and sizes in the corpus change over time.
- Chunking uniformly across every source type without checking whether the loader-level granularity already solved the problem for that source.
- Not tracing a "no relevant chunk found" failure back to chunking at all — assuming it's a retrieval ranking or embedding model problem first, when the actual chunks were incoherent or too large from the start.

### Lead-Level Interview Questions

**Q: Why is chunking necessary even for an LLM with a very large context window?**
A: Context window size only solves Problem 1 (fitting text in). It does nothing for Problem 2 (retrieval precision) — even with unlimited context, embedding a 20-page document as one vector still produces a blurry, imprecise match for any specific question about one part of it. Chunking exists independently of context window size, for retrieval quality reasons alone.

**Q: A retrieval pipeline returns confidently wrong answers. Walk through how chunking specifically could be the root cause.**
A: If chunks are too large and multi-topic, their embeddings represent a blend of several ideas, so a query can match a chunk that's only tangentially related to the actual question, and the LLM ends up answering based on the wrong section of that chunk. If chunks are too small or cut mid-sentence, the retrieved text may be syntactically broken or missing the context needed to answer correctly even though the right general area was found. Sampling actual retrieved chunks for a known-bad query is the fastest way to confirm whether chunking, not the model, is the cause.

**Q: Why doesn't loader-level granularity (one Document per CSV row, whole file for text references) eliminate the need for chunking entirely?**
A: It eliminates the need for some sources, but not all. A single JSON page can still be long and cover multiple unrelated policy clauses — loader-level granularity solved the structural unit (one page = one Document), but didn't guarantee that unit is short or single-topic. Chunking is the safety net for sources where the natural loader-level unit still isn't small or focused enough.

### Hidden Concepts Worth Knowing

- **Token count vs. character count**: context window limits are usually expressed in tokens, not characters or words, and the relationship between them varies by language and content (code, non-English text, and rare words typically use more tokens per character). A chunking strategy that only counts characters can still silently overflow a token-based limit.
- **The "lost in the middle" effect**: even when content technically fits in context, LLMs are empirically less accurate at using information placed in the middle of a long context versus the beginning or end — another reason smaller, focused chunks tend to outperform one giant context dump even when the giant dump would technically fit.

### Revision Summary

> Chunking solves two separate problems at once: fitting text within context window limits, and keeping each retrievable unit focused enough that its embedding actually represents one coherent idea. It's a second, independent splitting step on top of whatever granularity ingestion already chose — necessary wherever a Document is still too large or multi-topic after loading. This project's `chunk_text()` is sentence-aware but currently has zero overlap, a gap covered next.

## Topic 2: Chunk Size & Overlap

### Concept, Intuition, Why It Exists

- Chunk size and overlap are the two knobs that control the trade-off between coherence and context-loss at chunk boundaries.
- **Chunk size** is straightforward in concept but has competing pressures in practice: too large, and a chunk drifts back toward the blurry-embedding problem from the previous topic; too small, and a chunk loses the surrounding context that gives it meaning — a sentence like "this also applies to senior citizens" is meaningless on its own if the chunk before it (which defined what "this" refers to) got cut off.
- **Overlap** exists specifically to soften the chunk-boundary problem: instead of chunk N ending exactly where chunk N+1 begins, the last portion of chunk N is repeated as the first portion of chunk N+1. This means a sentence or idea that happens to fall near a boundary isn't fully lost to one side — it appears, at least partially, in both neighboring chunks, so whichever one gets retrieved still carries enough context to be useful.
- **This project's `chunk_text()` currently has zero overlap** — a real, named gap, not a stylistic choice. Every chunk boundary is a hard cut with nothing carried forward, meaning any idea that happens to span exactly across two chunks gets fragmented in both directions with no chunk containing the complete thought.

### Internal Working

- With overlap, chunking no longer advances by exactly `chunk_size` characters/sentences per step — it advances by `chunk_size - overlap`, so each new chunk re-includes the trailing `overlap` portion of the previous one.
- Concretely, for sentence-aware chunking: instead of starting the next chunk at the sentence immediately after where the last chunk ended, the next chunk starts a few sentences *before* that boundary — repeating the last `overlap` sentences (or characters) of the previous chunk before continuing with new content.
- This means the same piece of text near a boundary can now appear in two chunks' embeddings — a deliberate redundancy traded for safety against boundary-cut meaning loss.

### How It's Implemented In This Project

- The fix layers an `overlap` parameter onto the existing sentence-aware logic: after filling a chunk up to `chunk_size`, instead of starting completely fresh, the next chunk's starting point steps back by `overlap` sentences (or characters) before continuing forward — see the code below.

### Real-World Issues, Edge Cases, Debugging

- **Zero overlap's actual failure mode**: a policy clause like "premature withdrawal incurs a 1% penalty. This does not apply if the FD is closed due to the death of the depositor." — if the chunk boundary happens to land between these two sentences, one chunk says "1% penalty" with no exception mentioned, and the other says "this does not apply" with no idea what "this" refers to. A query about the death-of-depositor exception could retrieve either half and give an incomplete or actively misleading answer.
- **Overlap isn't free**: more overlap means more redundant text re-embedded and re-stored, directly increasing embedding cost and vector store size for the same source corpus. There's a real budget trade-off, not just a quality dial to crank up.
- **Choosing chunk size is itself content-dependent**: a dense policy clause needs a different chunk size than a conversational FAQ answer — a single fixed chunk size across all sources is a simplification, not a guarantee of correctness for every content type.
- **Debugging boundary loss**: if a known fact is in the source document but retrieval consistently misses it, check whether that fact happens to sit exactly at a chunk boundary before assuming the embedding or retrieval ranking is at fault — this is a very common, very specific root cause.

### Design Decisions & Trade-offs

- Overlap measured in sentences vs. characters: sentence-based overlap (carry forward the last N sentences) keeps boundaries coherent the same way sentence-aware chunking does in the first place; character-based overlap is simpler to implement but can re-cut mid-sentence at the new boundary, partially undermining the reason sentence-aware chunking was chosen.
- Fixed overlap value vs. content-aware overlap: a constant number of sentences/characters is simple and predictable; a smarter approach could vary overlap based on detected sentence boundary density or topic shifts, but adds real complexity for a marginal gain at this project's current scale.

### Alternatives & When To Use Each

- Zero overlap — acceptable only when source content has very low information density per sentence and ideas are unlikely to span exactly across a chunk boundary; risky as a default.
- Fixed sentence-count overlap (the fix here) — good general-purpose default, cheap to implement, directly addresses the boundary-cut problem.
- Fixed character-count overlap — simpler to implement than sentence-counting, but can still cut mid-sentence at the new chunk's start.
- Sliding-window overlap with no discrete chunk boundaries at all — maximizes boundary safety, but multiplies storage/embedding cost significantly since nearly all content gets embedded more than once; rarely worth it outside specialized high-recall use cases.

### Common Mistakes & Production Failures

- Shipping a chunker with zero overlap and discovering boundary-cut information loss only after a customer-facing answer misses an exception clause that was technically in the source document.
- Setting overlap too large relative to chunk size, causing massive redundancy where most of each chunk is just a repeat of its neighbor, inflating cost without meaningfully improving retrieval.
- Picking one global chunk size for every source type without checking whether dense policy text and conversational FAQ content actually need different sizes.

### Lead-Level Interview Questions

**Q: Why does zero overlap matter in practice, given that sentence-aware chunking already avoids cutting mid-sentence?**
A: Sentence-aware chunking guarantees individual sentences aren't broken, but says nothing about *related* sentences that happen to straddle a chunk boundary — an exception clause referencing "this" from the sentence before it can still be split across two chunks, each missing half the meaning. Overlap exists specifically to soften that cross-sentence boundary loss, which sentence-awareness alone doesn't solve.

**Q: How would you decide how much overlap to use, rather than picking an arbitrary number?**
A: Tie it to the typical length of a complete idea in your content — if exception clauses or related statements in this corpus typically span two to three sentences, overlap should cover at least that many sentences so a boundary cut never fully separates a complete thought. It should be validated against real retrieval failures (facts known to be in the source but missed at query time), not chosen purely theoretically.

**Q: What's the actual cost of adding overlap, and how would you reason about whether it's worth it?**
A: Overlap means some text gets embedded and stored more than once, increasing embedding calls and vector store size proportionally to the overlap fraction. It's worth it when boundary-cut information loss is a measured, real problem — confirmed by retrieval misses traceable to chunk boundaries — not applied reflexively as a default without checking whether the corpus actually has content dense enough to need it.

### Hidden Concepts Worth Knowing

- **The boundary-cut problem is a specific instance of a general windowing trade-off**: any system that processes a continuous stream in fixed windows (time-series analysis, audio processing, log analysis) faces the same tension — overlap trades redundancy for safety against information that happens to span a window edge.
- **Overlap doesn't fully solve the problem, it just reduces its frequency**: a sufficiently long related passage can still span beyond even a generous overlap window. Overlap shrinks the failure rate, it doesn't eliminate the failure mode entirely — worth stating honestly rather than treating it as a complete fix.

### Revision Summary

> Chunk size and overlap jointly control the coherence-vs-context-loss trade-off at chunk boundaries. This project's `chunk_text()` currently has zero overlap — a real gap where related ideas spanning a chunk boundary can be split with neither resulting chunk containing the complete thought. Adding overlap (carrying forward the last N sentences into the next chunk) directly addresses this, at the cost of some redundant embedding/storage — a trade that should be sized against real content, not picked arbitrarily.

In [3]:
"""
chunker.py
-----------
Sentence-aware chunking with configurable overlap.

Splits a Document's page_content into smaller Documents, never cutting
mid-sentence, and -- unlike a zero-overlap chunker -- carries forward the
last `overlap` sentences of each chunk into the start of the next one, so
an idea spanning a chunk boundary isn't fully lost to one side.
"""

import re
from document import make_document

SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def split_sentences(text: str) -> list:
    """Naive sentence splitter -- splits after ./!/? followed by whitespace.
    Good enough for structured policy/FAQ text; a real NLP sentence
    tokenizer (e.g. spaCy) would handle abbreviations more robustly."""
    sentences = SENTENCE_SPLIT_RE.split(text.strip())
    return [s.strip() for s in sentences if s.strip()]


def chunk_text(text: str, chunk_size: int = 500, overlap: int = 1) -> list:
    """Sentence-aware chunking with overlap.

    chunk_size : max characters per chunk (soft limit -- a chunk stops
                 adding sentences once it would exceed this length).
    overlap    : number of trailing sentences from the previous chunk to
                 repeat at the start of the next chunk. 0 = no overlap
                 (the old, gap-having behavior).

    IMPORTANT: overlap is automatically capped so it can never consume an
    entire completed chunk. If it did, carrying back "the last N sentences"
    would just rebuild the identical chunk every time and the loop would
    never make progress -- a real infinite-loop bug, not just slowness.
    """
    sentences = split_sentences(text)
    if not sentences:
        return []

    chunks = []
    current = []
    current_len = 0
    i = 0

    while i < len(sentences):
        sentence = sentences[i]
        if current_len + len(sentence) > chunk_size and current:
            chunks.append(" ".join(current))

            # Cap overlap so at least ONE sentence is always dropped --
            # this is what guarantees forward progress.
            safe_overlap = min(overlap, len(current) - 1) if overlap > 0 else 0
            carry_back = current[-safe_overlap:] if safe_overlap > 0 else []

            current = list(carry_back)
            current_len = sum(len(s) for s in current)
            continue  # re-evaluate the same sentence against the new chunk
        current.append(sentence)
        current_len += len(sentence)
        i += 1

    if current:
        chunks.append(" ".join(current))

    return chunks


def chunk_document(doc: dict, chunk_size: int = 500, overlap: int = 1) -> list:
    """Splits one Document into several smaller Documents, preserving and
    extending metadata with a chunk index so each piece is still traceable
    back to its source/page/row."""
    pieces = chunk_text(doc["page_content"], chunk_size=chunk_size, overlap=overlap)
    result = []
    for idx, piece in enumerate(pieces):
        meta = dict(doc["metadata"])
        meta["chunk_index"] = idx
        result.append(make_document(page_content=piece, metadata=meta))
    return result


if __name__ == "__main__":
    sample = (
        "Premature withdrawal incurs a 1 percent penalty on the applicable rate. "
        "This does not apply if the FD is closed due to the death of the depositor. "
        "In such cases, the full contracted interest rate is paid up to the date of closure. "
        "Senior citizens receive an additional 0.5 percent interest on all tenures. "
        "This additional rate applies only to resident senior citizens aged 60 and above."
    )

    # chunk_size bumped to 180 so each chunk can actually fit 2 sentences --
    # at 120, no chunk could ever hold more than 1 sentence, so overlap had
    # nothing to carry forward and silently did nothing.
    CHUNK_SIZE = 180

    print("--- Zero overlap (the old gap) ---")
    for i, c in enumerate(chunk_text(sample, chunk_size=CHUNK_SIZE, overlap=0)):
        print(f"  Chunk {i}: {c}")

    print("\n--- With overlap=1 (the fix) ---")
    for i, c in enumerate(chunk_text(sample, chunk_size=CHUNK_SIZE, overlap=1)):
        print(f"  Chunk {i}: {c}")

    print("\n--- As Documents (chunk_document) ---")
    doc = make_document(sample, {"source": "fd_policy_demo.txt", "page": 1})
    for d in chunk_document(doc, chunk_size=CHUNK_SIZE, overlap=1):
        print(f"  {d['metadata']}: {d['page_content'][:60]!r}...")

--- Zero overlap (the old gap) ---
  Chunk 0: Premature withdrawal incurs a 1 percent penalty on the applicable rate. This does not apply if the FD is closed due to the death of the depositor.
  Chunk 1: In such cases, the full contracted interest rate is paid up to the date of closure. Senior citizens receive an additional 0.5 percent interest on all tenures.
  Chunk 2: This additional rate applies only to resident senior citizens aged 60 and above.

--- With overlap=1 (the fix) ---
  Chunk 0: Premature withdrawal incurs a 1 percent penalty on the applicable rate. This does not apply if the FD is closed due to the death of the depositor.
  Chunk 1: This does not apply if the FD is closed due to the death of the depositor. In such cases, the full contracted interest rate is paid up to the date of closure.
  Chunk 2: In such cases, the full contracted interest rate is paid up to the date of closure. Senior citizens receive an additional 0.5 percent interest on all tenures.
  Chunk 3: Se

## Topic 3: Chunking Strategy Comparison

### Concept, Intuition, Why It Exists

- "Chunking" isn't one algorithm — it's a family of strategies that all solve the same two problems from the first topic (context limits, retrieval precision) but make very different trade-offs about *where* to cut.
- **Fixed-size chunking**: cuts every N characters or tokens, with no regard for sentence or word boundaries at all. The simplest possible strategy, and the baseline every other strategy improves on.
- **Sentence-aware chunking (what this project has)**: cuts only at sentence boundaries, accumulating sentences until adding the next one would exceed a size limit. Never breaks a sentence in half, but still has no awareness of whether consecutive sentences are actually about the same topic.
- **Semantic chunking**: goes a step further than sentence-awareness — instead of cutting purely on a size limit, it cuts where the *meaning* shifts. Concretely, this usually means embedding consecutive sentences and watching for a drop in similarity between neighboring sentences; a big drop signals a topic change, which becomes the cut point regardless of accumulated size.
- **Structure-aware chunking (Q&A pairs)**: instead of deriving chunk boundaries from size or meaning-drift, it uses the document's own existing structure — a question and its answer, a numbered SOP step, a table row — as the natural, pre-existing chunk boundary. This is the strongest strategy when the source data already has clean internal structure to exploit, because it sidesteps the size-vs-meaning trade-off entirely: the document tells you where the boundaries are.

### Internal Working

- Fixed-size: track a running character/token count; cut the instant the limit is hit, regardless of what's mid-word or mid-sentence at that point.
- Sentence-aware: split the text into sentences first; accumulate sentences into a chunk until the next one would exceed the size limit, then start a new chunk (with optional overlap, from the previous topic).
- Semantic: split into sentences, embed each one, compute similarity between consecutive sentence embeddings, and cut wherever similarity drops below a threshold — chunk size becomes a secondary constraint (a cap), not the primary cutting signal.
- Structure-aware (Q&A): parse the document's known structure directly — e.g. regex or pattern matching for "Q:" / "A:" markers, or numbered list items — and emit one chunk per structural unit, with no size-based logic involved at all unless a single structural unit is itself too large.

### How It's Implemented In This Project

- The project currently uses sentence-aware chunking (`chunk_text()`, covered in the previous topic) as the general-purpose default for JSON-sourced PDF pages and similar prose content.
- Structure-aware chunking is a natural fit for any FAQ-style source content in this project — wherever a document already presents clean Q&A pairs, splitting on that structure directly outperforms running generic sentence-aware chunking over the same content and hoping size-based cuts happen to land between pairs.

### Real-World Issues, Edge Cases, Debugging

- **Fixed-size chunking's failure mode is visible immediately**: cutting mid-word or mid-sentence produces chunks that are syntactically broken, which both reads badly if ever shown to a user and embeds poorly, since the embedding model is processing a fragment, not a coherent unit.
- **Sentence-aware chunking's failure mode is subtler**: it guarantees sentence integrity but not topical integrity — a chunk can still legally contain three unrelated sentences just because they happened to fit under the size limit together, producing the same "blurry embedding" problem chunking was meant to solve in the first place, just less severely than fixed-size.
- **Semantic chunking's failure mode is cost and instability**: it requires embedding every sentence just to *decide* where to cut, before the real per-chunk embedding even happens — meaningfully more compute per document than sentence-aware chunking. The similarity-drop threshold is also a tuning problem with the same sensitivity issues as the near-duplicate threshold from an earlier topic — too sensitive, and it over-splits into too many tiny chunks; too lenient, and it barely improves on plain sentence-aware chunking.
- **Structure-aware chunking's failure mode is brittleness**: it only works as well as the structure-detection logic. A Q&A document with inconsistent formatting (some pairs marked "Q:"/"A:", others just formatted as bold text with no marker) will have some pairs correctly extracted and others silently missed or merged incorrectly — this strategy trades generality for precision, and breaks quietly when the assumed structure doesn't hold.

### Design Decisions & Trade-offs

- Sentence-aware as the project's default general-purpose strategy: meaningfully better than fixed-size for near-zero extra cost (just sentence splitting, no embeddings needed to decide cut points), even though it doesn't fully solve topical coherence the way semantic chunking would.
- Reserving structure-aware chunking specifically for content that has known, reliable internal structure (Q&A pairs), rather than trying to force one universal strategy across every source type — this mirrors the same "right granularity for the right source" principle used at the loader level for text/CSV/JSON.
- Semantic chunking deliberately not adopted as the default here: the added embedding cost and threshold-tuning complexity isn't justified unless sentence-aware chunking is measurably failing on this corpus — a trade-off worth revisiting if retrieval quality issues are specifically traced back to topically-mixed chunks.

### Alternatives & When To Use Each

- Fixed-size chunking — only when speed/simplicity matters far more than quality, or as a quick baseline to compare smarter strategies against.
- Sentence-aware chunking — solid general-purpose default for prose content (policy text, SOPs, FAQ answers as paragraphs) where no embeddings-at-chunk-time cost is wanted; this project's current default.
- Semantic chunking — worth the extra embedding cost when sentence-aware chunking is confirmed (not assumed) to be producing topically-mixed chunks that hurt retrieval precision, typically on long-form prose with frequent unmarked topic shifts.
- Structure-aware chunking — the best choice whenever source content already has reliable, consistent internal structure (Q&A pairs, numbered SOP steps, table rows) to exploit directly, since it sidesteps the size-vs-meaning trade-off entirely.

### Common Mistakes & Production Failures

- Defaulting to fixed-size chunking purely because it's the simplest to implement, without recognizing the syntactic-fragment cost it imposes on every chunk.
- Reaching for semantic chunking as a default "more advanced = better" choice without confirming sentence-aware chunking is actually the bottleneck — paying ongoing embedding cost for a problem that may not exist in this specific corpus.
- Applying generic sentence-aware chunking to clearly structured Q&A content instead of exploiting the existing structure directly, producing chunks that split a question from its own answer simply because they landed on opposite sides of a size-based cut.
- Assuming one chunking strategy must be used uniformly across the entire corpus, rather than choosing per source type the same way loader granularity was chosen per source type in earlier topics.

### Lead-Level Interview Questions

**Q: Sentence-aware chunking already avoids breaking sentences. Why would semantic chunking ever be worth the extra cost?**
A: Sentence-aware chunking guarantees syntactic integrity, not topical integrity — three unrelated sentences can still legally end up in the same chunk just because they fit the size budget together, producing a blurry, multi-topic embedding. Semantic chunking cuts on meaning shift instead of size, directly targeting topical coherence — worth the extra embedding-at-chunk-time cost specifically when retrieval quality issues are traced back to topically-mixed chunks, not as a default upgrade.

**Q: You have a clean FAQ document with consistent Q&A markers. Why not just run the standard sentence-aware chunker over it like everything else?**
A: Because the document already tells you exactly where the ideal chunk boundaries are — each Q&A pair is a complete, self-contained unit. Running a size-based chunker over it risks cutting a question away from its own answer purely because of where the size limit happened to land, which a structure-aware approach that splits directly on the Q&A markers avoids entirely, with less computational cost than even sentence-aware chunking since no sentence-boundary detection is needed.

**Q: What's the actual failure mode of structure-aware chunking, and how would you detect it in production?**
A: It only works as well as the structure-detection logic — inconsistent formatting in the source document (some Q&A pairs marked clearly, others not) causes silent extraction failures: pairs get missed or merged incorrectly, with no exception thrown. Detecting this requires sampling the actual extracted chunks against the source document and checking extraction completeness, the same "validate the output, don't trust no errors thrown" discipline that applies everywhere else in this pipeline.

**Q: How would you decide which chunking strategy to use for a brand-new source type added to this project?**
A: Start by asking whether the source already has reliable internal structure to exploit (structure-aware first choice if yes). If not, default to sentence-aware as the safe general-purpose baseline. Only escalate to semantic chunking if sentence-aware chunking is measured, not assumed, to be producing topically-incoherent chunks that hurt retrieval — escalating cost should follow evidence of a real problem, not a desire to use a more sophisticated technique.

### Hidden Concepts Worth Knowing

- **The cost-vs-quality ladder repeats across this entire sub-chapter**: fixed-size (cheapest, lowest quality) → sentence-aware (cheap, better) → semantic (expensive, best general-purpose quality) → structure-aware (cheapest of all when applicable, often the *highest* quality because it uses ground-truth structure instead of inferring it) — the same cost-ordering principle seen earlier with hash-before-embedding dedup applies here too: use the cheapest strategy that's actually sufficient for the content.
- **Hybrid strategies are common in real systems**: structure-aware extraction first (split on Q&A markers, SOP steps), falling back to sentence-aware chunking only for any remaining unstructured prose in the same document — not a strict either/or choice per document.

### Revision Summary

> Four chunking strategies trade cost against quality differently: fixed-size is cheap but breaks sentences; sentence-aware (this project's default) preserves sentences but not topical coherence; semantic chunking cuts on meaning-shift for better coherence at real embedding cost; structure-aware chunking exploits a document's own existing structure (Q&A pairs) for the best result at the lowest cost, but only where that structure reliably exists. Choose per source type, escalate to more expensive strategies only on evidence of a real problem.

In [1]:
"""
chunking_strategies.py
------------------------
Side-by-side implementations of four chunking strategies, so they can be
compared directly on the same input text.

  fixed_size_chunk     -- cheapest, lowest quality: cuts every N characters,
                           no regard for word/sentence boundaries.
  sentence_aware_chunk -- this project's default (see chunker.py): never
                           breaks a sentence, no awareness of topic shifts.
  semantic_chunk        -- cuts where consecutive-sentence similarity drops
                           below a threshold, i.e. where the topic changes.
  qa_structure_chunk    -- exploits existing "Q: ... A: ..." structure
                           directly; no size or meaning-based logic at all.
"""

import re
import numpy as np
from document import make_document
from chunker import split_sentences  # reuse the sentence splitter from Topic 2

_model = None


def _get_model():
    global _model
    if _model is None:
        from sentence_transformers import SentenceTransformer
        _model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    return _model


# ----------------------------------------------------------------------
# 1. Fixed-size -- cheapest, lowest quality
# ----------------------------------------------------------------------
def fixed_size_chunk(text: str, chunk_size: int = 120) -> list:
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]


# ----------------------------------------------------------------------
# 2. Sentence-aware -- this project's default (no overlap shown here;
#    see chunker.py for the overlap-enabled version)
# ----------------------------------------------------------------------
def sentence_aware_chunk(text: str, chunk_size: int = 120) -> list:
    sentences = split_sentences(text)
    chunks, current, current_len = [], [], 0
    for sentence in sentences:
        if current_len + len(sentence) > chunk_size and current:
            chunks.append(" ".join(current))
            current, current_len = [], 0
        current.append(sentence)
        current_len += len(sentence)
    if current:
        chunks.append(" ".join(current))
    return chunks


# ----------------------------------------------------------------------
# 3. Semantic -- cut where meaning shifts, not where size runs out
# ----------------------------------------------------------------------
def semantic_chunk(text: str, similarity_threshold: float = 0.5) -> list:
    sentences = split_sentences(text)
    if len(sentences) <= 1:
        return sentences

    model = _get_model()
    embeddings = model.encode(sentences, normalize_embeddings=True, show_progress_bar=False)

    chunks, current = [], [sentences[0]]
    for i in range(1, len(sentences)):
        similarity = float(np.dot(embeddings[i - 1], embeddings[i]))  # normalized -> dot == cosine
        if similarity < similarity_threshold:
            chunks.append(" ".join(current))
            current = []
        current.append(sentences[i])
    if current:
        chunks.append(" ".join(current))
    return chunks


# ----------------------------------------------------------------------
# 4. Structure-aware -- exploit existing Q&A markers directly
# ----------------------------------------------------------------------
QA_PATTERN = re.compile(r"Q:\s*(.+?)\s*A:\s*(.+?)(?=Q:|$)", re.DOTALL)


def qa_structure_chunk(text: str) -> list:
    """Returns one chunk per Q&A pair, formatted as 'Q: ... A: ...'."""
    pairs = QA_PATTERN.findall(text)
    return [f"Q: {q.strip()} A: {a.strip()}" for q, a in pairs]


if __name__ == "__main__":
    sample = (
        "Premature withdrawal incurs a 1 percent penalty on the applicable rate. "
        "This does not apply if the FD is closed due to the death of the depositor. "
        "Senior citizens receive an additional 0.5 percent interest on all tenures."
    )

    print("--- Fixed-size ---")
    for c in fixed_size_chunk(sample, chunk_size=60):
        print(f"  {c!r}")

    print("\n--- Sentence-aware ---")
    for c in sentence_aware_chunk(sample, chunk_size=120):
        print(f"  {c!r}")

    print("\n--- Semantic ---")
    for c in semantic_chunk(sample, similarity_threshold=0.3):
        print(f"  {c!r}")

    qa_sample = (
        "Q: What is the penalty for premature withdrawal? "
        "A: A 1 percent penalty on the applicable rate. "
        "Q: Do senior citizens get a higher rate? "
        "A: Yes, an additional 0.5 percent on all tenures."
    )
    print("\n--- Structure-aware (Q&A) ---")
    for c in qa_structure_chunk(qa_sample):
        print(f"  {c!r}")

--- Fixed-size ---
  'Premature withdrawal incurs a 1 percent penalty on the appli'
  'cable rate. This does not apply if the FD is closed due to t'
  'he death of the depositor. Senior citizens receive an additi'
  'onal 0.5 percent interest on all tenures.'

--- Sentence-aware ---
  'Premature withdrawal incurs a 1 percent penalty on the applicable rate.'
  'This does not apply if the FD is closed due to the death of the depositor.'
  'Senior citizens receive an additional 0.5 percent interest on all tenures.'

--- Semantic ---

  'Premature withdrawal incurs a 1 percent penalty on the applicable rate.'
  'This does not apply if the FD is closed due to the death of the depositor.'
  'Senior citizens receive an additional 0.5 percent interest on all tenures.'

--- Structure-aware (Q&A) ---
  'Q: What is the penalty for premature withdrawal? A: A 1 percent penalty on the applicable rate.'
  'Q: Do senior citizens get a higher rate? A: Yes, an additional 0.5 percent on all tenures.'


## Topic 4: Different Types of Chunking & Text Splitters

### Concept, Intuition, Why It Exists

- Topic 3 covered chunking *strategies* (fixed-size, sentence-aware, semantic, structure-aware) as conceptual approaches. This topic covers **text splitters** — the actual reusable, named implementations of those strategies that show up repeatedly across RAG tooling (LangChain and similar libraries), worth knowing by name because interviewers and real codebases reference them directly, not just the underlying concept.
- A "splitter" is just a specific, parameterized implementation of a chunking strategy, usually built to be swappable — same input Document in, different splitter, different chunk boundaries out, with no other part of the pipeline needing to change. This is the same Document-abstraction payoff as the loaders: splitters are interchangeable precisely because they all consume and produce the same shape.
- **CharacterTextSplitter**: splits on a single fixed separator (commonly a blank line or newline) once content exceeds chunk size, falling back to a hard character cut if that separator never appears within the limit. It's the library-level cousin of fixed-size chunking from Topic 3, with one specific improvement — it tries a separator before giving up and cutting blindly.
- **RecursiveCharacterTextSplitter**: tries a *list* of separators in priority order, falling back from the most meaning-preserving separator to the least. Covered in full depth in the next topic, since it's the most commonly used default in practice and deserves its own dedicated treatment.
- **TokenTextSplitter**: splits based on actual token count (using the same tokenizer the target LLM/embedding model uses) rather than character count. This directly addresses the "token count vs. character count" hidden concept flagged back in Topic 1 — a chunk that looks fine by character count can still silently overflow a token-based context limit, and this splitter is built specifically to prevent that.
- **MarkdownHeaderTextSplitter** (and equivalents for HTML, code, etc.): a structure-aware splitter, but generalized beyond just Q&A pairs — it splits on a document's own formatting markers (`#`, `##`, `###` headers in Markdown; HTML tags; code function/class boundaries) the same way Topic 3's Q&A splitter exploited "Q:"/"A:" markers, just for a different kind of structure.

### Internal Working

- **CharacterTextSplitter**: scan the text for the configured separator; accumulate content between separator occurrences until the next addition would exceed `chunk_size`; if no separator occurrence exists within the size budget at all, cut at the raw character limit as a last resort.
- **TokenTextSplitter**: tokenize the full text first using the target tokenizer, then group tokens into chunks of the configured token count, decoding each group back into text — this guarantees every chunk respects the token budget exactly, unlike character-based splitters which only approximate it.
- **MarkdownHeaderTextSplitter**: parse the document for header markers, treat each header's section as a structural unit, and attach the header hierarchy itself into each chunk's metadata (e.g. "this chunk falls under Section 2 > Subsection 2.1") — giving retrieval extra structural context beyond just the chunk text itself, similar in spirit to how the JSON loader attaches `document_code`/`version`/`page` metadata.

### How It's Implemented In This Project

- This project's hand-written `chunk_text()`/`chunker.py` (Topic 2) is functionally closest to a sentence-aware variant of CharacterTextSplitter, built by hand rather than via a library, with overlap added explicitly.
- TokenTextSplitter-style logic becomes directly relevant the moment chunk sizing needs to be tied precisely to the embedding model's or LLM's actual token limit rather than an approximate character count — worth adopting if/when this project starts hitting silent token-limit truncation in production.
- MarkdownHeaderTextSplitter-style logic is directly applicable to any future Markdown-formatted SOPs or internal docs ingested into this project, the same way the Q&A structure-aware splitter applies to FAQ content — both are instances of "exploit existing document structure instead of inferring boundaries."

### Real-World Issues, Edge Cases, Debugging

- **CharacterTextSplitter's fallback is a silent quality cliff**: when the configured separator never appears within the chunk size budget (e.g. one extremely long paragraph with no blank lines), it falls back to a hard character cut with zero warning — producing exactly the syntactically-broken chunks this whole topic exists to avoid, but only for the specific documents unlucky enough to trigger the fallback. Worth explicitly checking for and flagging, not assuming the splitter always behaves the better way.
- **TokenTextSplitter requires picking the *right* tokenizer**: using a different model's tokenizer than the one actually being used downstream produces a chunk that's correctly sized for the wrong model — e.g. sizing chunks using one model's tokenizer while embedding with a completely different model's tokenizer means the resulting token counts can be meaningfully off for the model that actually matters.
- **MarkdownHeaderTextSplitter assumes well-formed, consistent headers**: a document with inconsistent header levels (skipping from `#` straight to `###`, or using bold text instead of real header markers in places) produces an inconsistent or incomplete structural hierarchy — the same brittleness failure mode flagged for Q&A structure-aware chunking in Topic 3, just applied to a different structural cue.
- **Mixing splitter types within one pipeline**: nothing prevents using different splitters for different source types in the same project (Markdown docs with the header splitter, FAQ content with the Q&A splitter, generic prose with the recursive splitter) — this is the recommended approach, not a complication to avoid, mirroring the same per-source-type decision already made at the loader level.

### Design Decisions & Trade-offs

- Library-provided splitters vs. hand-written chunking logic: this project currently hand-writes its chunker, trading a dependency for full visibility and control over exact behavior — reasonable while the team wants to understand every step deeply (as this whole learning sequence demonstrates), but a real production system more often reaches for well-tested library splitters to avoid re-solving already-solved edge cases (tokenizer alignment, separator fallback behavior) from scratch.
- Token-based vs. character-based sizing: token-based is strictly more correct for respecting an LLM's actual context limit, but requires running the tokenizer at chunk time, which is a real (if usually small) additional compute cost compared to just counting characters.

### Alternatives & When To Use Each

- CharacterTextSplitter — simple prose with a reliable separator (consistent paragraph breaks) and a tolerance for occasional fallback-cut chunks on unusually long sections.
- TokenTextSplitter — chunk size needs to map precisely to a model's actual context/embedding limit, not an approximation; worth the small extra tokenization cost when token-limit overflows are a measured real problem.
- MarkdownHeaderTextSplitter (or HTML/code equivalents) — source content already has reliable, consistent structural markers (headers, tags, function boundaries) to exploit directly, the same reasoning as Q&A structure-aware chunking in Topic 3.
- Hand-written custom logic (this project's current approach) — full control and visibility needed, team is building deep understanding of the mechanics rather than treating chunking as a black box.

### Common Mistakes & Production Failures

- Assuming character-based chunk sizes translate predictably to token counts, especially for non-English content or code, where the character-to-token ratio shifts significantly.
- Using a tokenizer mismatched to the actual downstream model when token-based splitting is adopted, producing chunks sized correctly for the wrong model.
- Applying a structure-aware splitter (Markdown headers, Q&A) to content that doesn't actually have consistent structure, and not validating extraction completeness afterward.
- Treating "which splitter to use" as a one-time, project-wide decision instead of a per-source-type decision, the same mistake flagged at the strategy level in Topic 3.

### Lead-Level Interview Questions

**Q: Why would you ever need a token-based splitter instead of just being more conservative with character-based chunk sizes?**
A: Because the character-to-token ratio isn't fixed — it varies by language, content type, and even specific vocabulary. Being "conservative" with character counts to be safe means either wasting context budget on content that tokenizes efficiently, or still risking overflow on content that tokenizes poorly (dense technical text, non-English content, code) — a token-based splitter removes the guesswork entirely by measuring the thing that actually matters.

**Q: A document's CharacterTextSplitter-based chunking silently falls back to hard character cuts on a few documents. How would you detect this happening in production without manually reading every chunk?**
A: Track, per document, whether the configured separator was actually found within the size budget or whether the fallback path was taken — log a flag or metric whenever the fallback fires, then monitor that count the same way other ingestion stages monitor warning counts. A spike signals specific documents (or document types) where the separator assumption doesn't hold, worth investigating rather than letting silently degrade.

**Q: When would you choose a Markdown/structure-aware splitter over a generic recursive splitter, even though the recursive splitter is more broadly applicable?**
A: When the source content has reliable, consistent structural markers that directly encode the ideal chunk boundaries — splitting on real headers preserves the document's own logical organization (and lets that hierarchy be attached as metadata for richer retrieval context) far more precisely than inferring boundaries purely from separators or size, the same reasoning that favored Q&A structure-aware chunking over generic chunking in Topic 3.

### Hidden Concepts Worth Knowing

- **Tokenizer alignment as a cross-cutting concern**: the same token-vs-character mismatch that affects chunk sizing here also affects cost estimation, context-window budgeting at query time, and prompt construction generally — it's not a chunking-specific issue, just one of the places it shows up first.
- **Metadata-enriched chunks**: structure-aware splitters that attach header hierarchy or section context into chunk metadata enable retrieval-time filtering and citation generation beyond what plain text-similarity search alone provides — e.g. "only search within Section 3" or "show the user which section this answer came from," both of which depend on that structural metadata being captured at chunk time, not reconstructed later.

### Revision Summary

> Beyond conceptual chunking strategies, several named, reusable splitter implementations exist: CharacterTextSplitter (single separator with hard-cut fallback), TokenTextSplitter (sizes by actual tokenizer output, not character approximation), and MarkdownHeaderTextSplitter (generalizes structure-aware chunking to any document with formatting markers, attaching structural metadata along the way). RecursiveCharacterTextSplitter — the most commonly used default — gets its own dedicated topic next.

In [ ]:
"""
text_splitters.py
--------------------
Hand-written implementations of three named splitter types, built to be
directly comparable to chunker.py's sentence-aware splitter and the four
strategies in chunking_strategies.py.

  character_text_split  -- single separator, hard-cut fallback.
  token_text_split        -- sizes chunks by actual token count, not chars.
  markdown_header_split   -- splits on Markdown headers, attaching the
                              header hierarchy into each chunk's metadata.
"""

import re
from document import make_document


# ----------------------------------------------------------------------
# 1. CharacterTextSplitter -- single separator, hard-cut fallback
# ----------------------------------------------------------------------
def character_text_split(text: str, separator: str = "\n\n", chunk_size: int = 200) -> list:
    pieces = text.split(separator)
    chunks, current = [], ""

    for piece in pieces:
        candidate = (current + separator + piece) if current else piece
        if len(candidate) > chunk_size:
            if current:
                chunks.append(current)
            if len(piece) > chunk_size:
                # fallback: separator didn't help, hard-cut this piece
                for i in range(0, len(piece), chunk_size):
                    chunks.append(piece[i:i + chunk_size])
                current = ""
            else:
                current = piece
        else:
            current = candidate

    if current:
        chunks.append(current)
    return chunks


# ----------------------------------------------------------------------
# 2. TokenTextSplitter -- sizes by token count, not character count.
#    Uses a simple whitespace tokenizer as a stand-in; swap in a real
#    tokenizer (e.g. tiktoken, or the embedding model's own tokenizer)
#    to match this exactly to your actual downstream model.
# ----------------------------------------------------------------------
def _simple_tokenize(text: str) -> list:
    return text.split()  # placeholder tokenizer -- replace with a real one in production


def _simple_detokenize(tokens: list) -> str:
    return " ".join(tokens)


def token_text_split(text: str, chunk_size_tokens: int = 30) -> list:
    tokens = _simple_tokenize(text)
    return [
        _simple_detokenize(tokens[i:i + chunk_size_tokens])
        for i in range(0, len(tokens), chunk_size_tokens)
    ]


# ----------------------------------------------------------------------
# 3. MarkdownHeaderTextSplitter -- splits on # / ## / ### headers,
#    attaching the header hierarchy into each chunk's metadata.
# ----------------------------------------------------------------------
HEADER_RE = re.compile(r"^(#{1,3})\s+(.*)$", re.MULTILINE)


def markdown_header_split(text: str) -> list:
    matches = list(HEADER_RE.finditer(text))
    if not matches:
        return [make_document(text.strip(), {"header_path": []})]

    documents = []
    header_stack = []  # tracks current (level, title) hierarchy

    for i, match in enumerate(matches):
        level = len(match.group(1))
        title = match.group(2).strip()

        # pop any headers at the same or deeper level before pushing this one
        header_stack = [h for h in header_stack if h[0] < level]
        header_stack.append((level, title))

        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        section_text = text[start:end].strip()

        if section_text:
            documents.append(make_document(
                page_content=section_text,
                metadata={"header_path": [h[1] for h in header_stack]},
            ))

    return documents


if __name__ == "__main__":
    sample = (
        "Premature withdrawal incurs a 1 percent penalty.\n\n"
        "This does not apply if the FD is closed due to the death of the depositor.\n\n"
        "Senior citizens receive an additional 0.5 percent interest on all tenures."
    )

    print("--- CharacterTextSplitter ---")
    for c in character_text_split(sample, separator="\n\n", chunk_size=80):
        print(f"  {c!r}")

    print("\n--- TokenTextSplitter ---")
    for c in token_text_split(sample, chunk_size_tokens=10):
        print(f"  {c!r}")

    markdown_sample = (
        "# FD Policy\n"
        "Overview text about FD policy.\n"
        "## Premature Withdrawal\n"
        "A 1 percent penalty applies.\n"
        "### Death of Depositor Exception\n"
        "No penalty applies in this case.\n"
        "## Senior Citizen Rate\n"
        "An additional 0.5 percent applies.\n"
    )
    print("\n--- MarkdownHeaderTextSplitter ---")
    for d in markdown_header_split(markdown_sample):
        print(f"  header_path={d['metadata']['header_path']}: {d['page_content']!r}")

## Topic 5: RecursiveCharacterTextSplitter Mechanics

### Concept, Intuition, Why It Exists

- CharacterTextSplitter (Topic 4) has one weakness baked into its design: it commits to a *single* separator, and the instant that separator doesn't appear within the size budget, it falls back straight to a hard character cut — there's no middle ground between "clean separator-based split" and "blind character chop."
- **RecursiveCharacterTextSplitter** fixes exactly this by trying a *list* of separators, in priority order, instead of just one — falling back progressively from the most meaning-preserving separator to the least, rather than jumping straight from "best case" to "worst case."
- The default separator priority order is: **paragraph break → line break → space → character**. This order isn't arbitrary — it's a deliberate ranking from "most likely to preserve a complete idea" to "least likely," and the splitter only drops down a level when the current level genuinely can't produce a small-enough piece.
- This is the same cost-ordering philosophy seen repeatedly across earlier topics — hash-before-embedding in dedup, structure-aware-before-generic in chunking strategy choice — applied here as: try-the-best-option-before-falling-back, at every single split decision, not just once globally.

### Internal Working

The splitter processes text **recursively**, one separator level at a time:

1. Start with the first separator in the priority list — paragraph break (typically a double newline).
2. Split the text on that separator. For each resulting piece: if it already fits within `chunk_size`, keep it as a chunk and move on. If it's still too large, **don't fall back to a hard cut immediately** — instead, recursively re-apply the splitting process to that oversized piece using the *next* separator down the priority list.
3. This continues down the chain: a paragraph too big gets split by line breaks; a line too big gets split by spaces (i.e., word boundaries); a single "word" or run of text too big even for that (rare, but possible with no whitespace at all) finally gets hard-cut by character count as the absolute last resort.
4. Adjacent small pieces produced by a given separator level get **merged back together** up to `chunk_size` before moving on — the splitter doesn't emit one chunk per individual line or sentence if several of them comfortably fit together under the size limit; it greedily packs them, the same accumulate-until-the-limit logic Topic 2's sentence-aware chunker uses, just generalized across separator levels.

### Why Paragraph → Line → Space → Character Is the Default Order

- **Paragraph break first**: a paragraph is, in most well-written prose, the largest unit that's still reliably a complete, coherent idea — splitting here first means the *majority* of real documents never need to fall further down the chain at all.
- **Line break second**: when a paragraph alone is still too large (long policy text, dense technical content), individual lines are the next-most-coherent fallback — closer to a complete thought than an arbitrary word-count cut, especially for structured content like bullet points or numbered clauses that are often one per line.
- **Space (word boundary) third**: if even a single line is too long, splitting on spaces guarantees the cut never lands mid-word — it sacrifices sentence/line coherence but still preserves the smallest meaningful linguistic unit, a word.
- **Character last, absolute fallback**: only triggered when a single "word" (or a run of text with no whitespace at all — a long ID string, a URL, a block of code with no spaces) is itself larger than the chunk size. This is the only level where a cut can land somewhere genuinely meaningless, and it's reached only when every more meaning-preserving option has already failed.
- The order is, in short, a ranked bet: try the split that's most likely to preserve meaning first, and only pay the cost of a worse split when the better option provably doesn't work for that specific piece of text.

### How It's Implemented In This Project

- This project's hand-written sentence-aware chunker (Topic 2) is a simpler, single-level version of this same idea — it commits to "sentence" as its one splitting unit, the way CharacterTextSplitter commits to one separator, without the recursive fallback chain.
- Adopting full recursive splitting becomes worthwhile the moment source content stops being reliably well-formed prose — e.g. ingesting raw SOP text with inconsistent paragraph spacing, where sentence-only splitting might occasionally hit a single "sentence" (really an unbroken run of clauses with no terminal punctuation) too large to fit the size budget, with nowhere to fall back to.

### Real-World Issues, Edge Cases, Debugging

- **Recursion depth matters for content with unusual structure**: code blocks, tables rendered as plain text, or extremely long unbroken strings (IDs, URLs) can force the splitter all the way down to the character-level fallback — worth explicitly checking which separator level actually produced each chunk in production, since chunks from the character-level fallback are exactly the syntactically-broken pieces every other strategy in this sub-chapter has been trying to avoid.
- **Merging behavior can still produce topically-mixed chunks**: because adjacent small pieces get packed together up to the size limit, two genuinely unrelated short paragraphs can still end up merged into one chunk purely because they fit — this splitter solves the "broken mid-sentence" problem completely, but doesn't solve the "topically blurry chunk" problem any better than plain sentence-aware chunking does; that's still semantic chunking's job if needed.
- **Custom separator lists for non-English or structured content**: the default paragraph/line/space/character order assumes whitespace-delimited language with clear paragraph breaks — content in a language without word-spacing, or content that's mostly tabular/structured rather than prose, needs a custom separator priority list rather than relying on defaults built around English-prose assumptions.

### Design Decisions & Trade-offs

- Recursive fallback vs. single-separator-then-hard-cut: more implementation complexity (a recursive function instead of a single split-and-pack loop) in exchange for meaningfully fewer character-level hard cuts across a real, varied corpus — worth the complexity the moment source content isn't uniformly well-formed.
- Default separator order vs. custom order per content type: the default order is a reasonable general-purpose bet for English prose, but isn't universally correct — code, tables, and non-English content may need their own tuned separator priority lists, the same per-source-type tuning principle seen throughout this sub-chapter.

### Alternatives & When To Use Each

- CharacterTextSplitter (Topic 4) — content is reliably well-formed with a consistent separator throughout, and the rare hard-cut fallback case is acceptable.
- RecursiveCharacterTextSplitter (this topic) — general-purpose default for mixed or imperfectly-formed prose; meaningfully reduces hard-cut fallback frequency at the cost of slightly more complex logic.
- Structure-aware splitting (Topic 3/4) — still the better choice whenever the content has known, reliable structure to exploit directly; recursive character splitting is a strong *fallback* for everything else, not a replacement for structure-aware splitting where structure genuinely exists.

### Common Mistakes & Production Failures

- Assuming RecursiveCharacterTextSplitter solves topical coherence because it sounds more sophisticated than sentence-aware chunking — it solves syntactic breakage far better, but says nothing about whether merged-together pieces are actually about the same topic.
- Using the default separator order unmodified on non-prose content (code, tabular text, non-English text without paragraph conventions) and getting worse results than a simpler, content-tuned splitter would have produced.
- Never checking which separator level actually produced a given chunk in production, missing a real signal that some documents are forcing frequent character-level fallback and need a different handling path entirely.

### Lead-Level Interview Questions

**Q: Why try multiple separators in priority order instead of just picking the one separator most likely to work, like CharacterTextSplitter does?**
A: No single separator is reliably present across every document and every piece of a document — a paragraph break might be missing in one section but present in another. Trying separators in priority order, recursively, means the splitter only pays the cost of a worse cut on the *specific pieces* that actually need it, rather than committing the entire document to one separator's worst case.

**Q: Walk through what happens when a single paragraph is too large even after splitting on line breaks and spaces.**
A: The splitter falls all the way to the character-level fallback for that specific piece, hard-cutting it regardless of word boundaries. This only happens for genuinely pathological content — an extremely long unbroken string with no internal whitespace at all (a URL, an ID, a run-on technical string) — and it's the one level where a cut can land somewhere truly meaningless, which is exactly why it's the last resort rather than an early option.

**Q: Does using RecursiveCharacterTextSplitter mean you no longer need to worry about topically-mixed chunks?**
A: No — it solves a different problem. It guarantees chunks aren't broken mid-word/mid-line/mid-paragraph more often than necessary, but two short, unrelated paragraphs can still get packed into the same chunk simply because they fit the size budget together. Topical coherence is what semantic chunking specifically targets; recursive character splitting and semantic chunking solve different problems and can be combined, not substituted for each other.

**Q: A new source type uses tables rendered as plain text with no real paragraph breaks. Would you use the default separator order?**
A: No — the default order assumes prose with reliable paragraph/line/space structure, which tabular plain text doesn't have. A custom separator priority list tuned to that content's actual structure (e.g. splitting on row delimiters first) would produce far better results than forcing the English-prose-tuned default order onto fundamentally non-prose content.

### Hidden Concepts Worth Knowing

- **Recursion as a general fallback-chain pattern**: trying the best option first and recursively degrading to progressively worse-but-more-reliable options is a pattern that shows up well beyond chunking — retry logic with escalating strategies, graceful degradation in distributed systems, and fallback model routing (try the best model, fall back to a cheaper/faster one on failure) all follow the same shape.
- **Separator order is itself a tunable hyperparameter, not a fixed law**: nothing about "paragraph → line → space → character" is mathematically required — it's an empirically reasonable default for English prose, and treating it as the one correct order rather than a sensible starting point is a subtle trap.

### Revision Summary

> RecursiveCharacterTextSplitter improves on single-separator splitting by trying a priority-ordered list of separators — paragraph, then line, then space, then character — recursively falling back only when the current level genuinely can't produce a small-enough piece. The order is a deliberate bet: try the most meaning-preserving split first, pay for a worse one only when necessary. It solves syntactic breakage well, but not topical coherence — that remains semantic chunking's job, and the two are complementary, not substitutes.

In [ ]:
"""
recursive_text_splitter.py
-----------------------------
RecursiveCharacterTextSplitter mechanics: tries separators in priority
order -- paragraph -> line -> space -> character -- recursively falling
back to the next separator only when the current one can't produce a
small-enough piece.

Unlike CharacterTextSplitter (Topic 4), which commits to ONE separator and
hard-cuts the instant it fails, this tries progressively less-preserving
splits before ever resorting to a blind character cut.
"""

from document import make_document

DEFAULT_SEPARATORS = ["\n\n", "\n", " ", ""]  # paragraph, line, space, character


def _split_with_separator(text: str, separator: str) -> list:
    if separator == "":
        return list(text)  # character-level: every character is its own piece
    return text.split(separator)


def _recursive_split(text: str, separators: list, chunk_size: int) -> list:
    """Core recursive logic: try the first separator; for any piece still
    too large, recurse into it with the NEXT separator down the list."""
    if len(text) <= chunk_size:
        return [text]

    separator = separators[0]
    remaining_separators = separators[1:]
    pieces = _split_with_separator(text, separator)

    final_pieces = []
    for piece in pieces:
        if len(piece) <= chunk_size:
            final_pieces.append(piece)
        elif remaining_separators:
            # this piece is still too big -- recurse with the next separator
            final_pieces.extend(_recursive_split(piece, remaining_separators, chunk_size))
        else:
            # no separators left -- absolute last resort, hard character cut
            final_pieces.extend(
                piece[i:i + chunk_size] for i in range(0, len(piece), chunk_size)
            )

    return final_pieces


def _merge_small_pieces(pieces: list, separator: str, chunk_size: int) -> list:
    """Greedily packs adjacent small pieces back together up to chunk_size,
    so we don't emit one chunk per tiny line/sentence when several fit
    comfortably together."""
    merged, current = [], ""
    for piece in pieces:
        candidate = (current + separator + piece) if current else piece
        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if current:
                merged.append(current)
            current = piece
    if current:
        merged.append(current)
    return merged


def recursive_character_split(text: str, chunk_size: int = 200, separators: list = None) -> list:
    separators = separators or DEFAULT_SEPARATORS
    raw_pieces = _recursive_split(text, separators, chunk_size)
    # merge using the broadest separator that's still safe (paragraph break)
    return _merge_small_pieces(raw_pieces, separators[0], chunk_size)


if __name__ == "__main__":
    sample = (
        "Premature withdrawal incurs a 1 percent penalty on the applicable rate.\n\n"
        "This does not apply if the FD is closed due to the death of the depositor. "
        "In such cases, the full contracted interest rate is paid up to the date of closure.\n\n"
        "Senior citizens receive an additional 0.5 percent interest on all tenures. "
        "This additional rate applies only to resident senior citizens aged 60 and above."
    )

    print("--- Default separator order (paragraph -> line -> space -> char) ---")
    for i, c in enumerate(recursive_character_split(sample, chunk_size=120)):
        print(f"  Chunk {i} ({len(c)} chars): {c!r}")

    print("\n--- Custom separator order (forces deeper fallback for comparison) ---")
    for i, c in enumerate(recursive_character_split(sample, chunk_size=40, separators=["\n\n", " ", ""])):
        print(f"  Chunk {i} ({len(c)} chars): {c!r}")

## Topic 6: Other Types of Splitters & Examples

### Concept, Intuition, Why It Exists

- The previous topics covered the core, general-purpose splitters. Real ingestion pipelines eventually run into source content that doesn't fit prose, Q&A, or Markdown assumptions cleanly — code files, HTML pages, JSON-as-data (not JSON-as-page-export like this project already uses), and very long documents where even recursive character splitting still produces topically-mixed chunks. This topic rounds out the picture with the remaining splitter types worth knowing by name.
- **CodeTextSplitter**: a language-aware variant of recursive splitting where the separator priority list is built from a programming language's actual syntax (function/class boundaries, then blocks, then lines) instead of paragraph/line/space. The same recursive-fallback mechanics from the previous topic, just with separators chosen to respect code structure instead of prose structure.
- **HTMLHeaderTextSplitter**: the HTML sibling of the Markdown header splitter — splits on `<h1>`/`<h2>`/`<h3>` tags (or other structural tags) instead of `#`/`##`/`###`, attaching the same kind of header-hierarchy metadata to each chunk.
- **JSONSplitter (structured-data splitter)**: rather than treating JSON as text to split on characters, this walks the JSON's actual nested structure and emits one chunk per logical node/object, keeping each chunk valid and self-contained relative to its place in the structure — directly relevant to this project's own JSON loader if a future JSON source has deeply nested data rather than the current flat `pages` list.
- **Agentic/LLM-based chunking**: instead of any rule-based separator logic at all, an LLM itself is prompted to read a document and propose chunk boundaries based on its own understanding of where ideas naturally divide. The most expensive option by far — every document costs an LLM call just to be chunked — but can outperform every rule-based approach on genuinely irregular content where no separator-based or embedding-similarity heuristic captures the actual structure.

### Internal Working

- **CodeTextSplitter**: recursively splits using a separator list ordered specifically for the target language — e.g. for Python: class definition, then function definition, then logical block (if/for/while), then line, then character — falling back exactly the same way Topic 5's recursive splitter does, just with code-aware separators instead of prose-aware ones.
- **HTMLHeaderTextSplitter**: parses the HTML tree, identifies header tags at each level, and treats the content between one header and the next (within the same nesting level) as one structural unit — mechanically identical to the Markdown header splitter from Topic 4, operating on a different markup syntax.
- **JSONSplitter**: recursively walks the JSON object graph; for each key/value pair or array item, decides whether the serialized sub-structure fits within the size budget as one chunk, and if not, recurses one level deeper into that sub-structure — the same recursive divide-until-it-fits logic as Topic 5's character splitter, but operating on structural nesting instead of separator characters.
- **Agentic chunking**: send the full document (or large windows of it) to an LLM with a prompt asking it to return chunk boundaries (or pre-split text) based on topical coherence; parse the LLM's response back into discrete chunks. No separator logic at all — the LLM's own judgment replaces every rule used by every other splitter in this sub-chapter.

### How It's Implemented In This Project

- None of these are currently needed by this project's actual source types (text references, CSV/Excel rows, JSON page exports) — they're documented here because they're the natural next splitters to reach for the moment new source types are added: ingesting actual source code documentation would call for CodeTextSplitter; ingesting a scraped HTML knowledge base would call for HTMLHeaderTextSplitter; a future JSON source with deeply nested (not flat) structure would call for a JSON-structure-aware splitter instead of treating that JSON as flat text.
- Agentic chunking is explicitly **not** recommended as this project's default given its cost — flagged here mainly so it's recognized by name and reserved for genuinely irregular content where every cheaper option has been tried and measurably underperforms.

### Real-World Issues, Edge Cases, Debugging

- **CodeTextSplitter needs the correct language grammar to be useful at all** — using Python-tuned separators on a JavaScript file produces boundaries that don't actually align with that language's real structure, the same mismatch risk as using a mismatched tokenizer in Topic 4's TokenTextSplitter.
- **HTMLHeaderTextSplitter inherits the web's structural inconsistency** — real-world HTML is frequently malformed or uses headers purely for visual styling rather than genuine document structure (a `<h3>` used just because it "looks right" in a particular spot, with no real hierarchical meaning) — the same structure-reliability risk flagged for Markdown headers in Topic 4, just worse in practice because HTML in the wild is messier than authored Markdown.
- **JSONSplitter's chunk size vs. structural integrity trade-off**: forcing a strict size limit onto JSON can produce a chunk that's a syntactically valid but semantically incomplete fragment of a larger object (half of an array, with no indication the rest exists elsewhere) — worth deciding explicitly whether to allow oversized-but-complete chunks for some structures rather than always enforcing the size limit strictly.
- **Agentic chunking's biggest production risk is non-determinism and cost**: re-running the same document through an LLM-based chunker can produce slightly different boundaries each time, complicating the incremental-ingestion manifest pattern from the earlier sub-chapter (which assumes "same content in → same result out"), and the per-document LLM cost scales linearly with corpus size in a way none of the rule-based splitters do.

### Design Decisions & Trade-offs

- Rule-based splitters (everything except agentic chunking) trade some quality ceiling for predictability, speed, and cost — they're deterministic, fast, and free beyond the embedding cost already being paid anyway.
- Agentic chunking trades predictability and cost for a potentially higher quality ceiling on genuinely irregular content — only worth it when measured evidence shows rule-based splitters are failing on a specific content type, not adopted as a general-purpose default given its cost and non-determinism.
- Structure-aware splitters (Code, HTML, JSON) all share the same underlying trade-off already established for Markdown/Q&A splitters in earlier topics: highest quality when the assumed structure genuinely holds, brittle and silently degraded when it doesn't.

### Alternatives & When To Use Each

- CodeTextSplitter — ingesting source code or code-heavy technical documentation where function/class boundaries are the meaningful unit.
- HTMLHeaderTextSplitter — ingesting scraped or exported HTML content with genuine (not purely cosmetic) header structure.
- JSONSplitter — ingesting deeply nested structured JSON where flat text splitting would either break valid JSON syntax or ignore meaningful nesting.
- Agentic/LLM-based chunking — small volumes of genuinely irregular content where every cheaper, rule-based option has been tried and measurably underperforms; not a default for any high-volume pipeline.
- Recursive character splitting (Topic 5) — still the right general-purpose fallback for any content that doesn't cleanly fit one of the more specialized structures above.

### Common Mistakes & Production Failures

- Reaching for agentic chunking by default because it sounds like the most "advanced" option, without first confirming cheaper rule-based splitters are actually insufficient for the content.
- Using a structure-aware splitter (Code/HTML/JSON) on content where that structure is present but unreliable, and not validating chunk completeness afterward the same way every other structure-aware splitter in this sub-chapter has needed validation.
- Forgetting that non-deterministic chunking (agentic) breaks the "same input, same output" assumption that incremental ingestion's manifest pattern depends on — a model producing slightly different chunk boundaries on a re-run looks identical to the manifest (same file hash) but produces a different embedded result, a subtle mismatch worth being aware of before adopting it.

### Lead-Level Interview Questions

**Q: When would you justify the cost of agentic/LLM-based chunking over every rule-based alternative covered so far?**
A: Only after rule-based options (recursive character splitting, structure-aware splitting where applicable, semantic chunking) have been tried and shown, with actual evidence, to underperform on a specific content type — typically genuinely irregular content with no consistent separators, headers, or stable internal structure at all. It should never be a default; the cost (an LLM call per document) and non-determinism are real, ongoing costs, not one-time setup costs.

**Q: Why might using a structure-aware splitter on real-world scraped HTML be riskier than using one on hand-authored Markdown?**
A: Markdown headers are almost always used semantically — when someone writes `##`, they mean "this is a real subsection." HTML headers in scraped, real-world content are frequently used for visual styling rather than genuine document structure, so the same structure-aware splitting logic inherits a less reliable signal, producing structural metadata (and chunk boundaries) that look meaningful but may not actually reflect the document's real logical organization.

**Q: Agentic chunking re-run on the same document produces slightly different chunk boundaries each time. What does this break elsewhere in the pipeline?**
A: It breaks the implicit assumption behind the incremental ingestion manifest from the earlier sub-chapter — that manifest decides "skip re-ingestion" purely based on whether the source file's content hash changed, assuming the same input always deterministically produces the same chunked output. With non-deterministic chunking, a file with an unchanged hash could still, if reprocessed for any reason, produce a different embedded result than what's already in the vector store, an inconsistency rule-based splitters don't introduce.

### Hidden Concepts Worth Knowing

- **Determinism as a cross-cutting pipeline requirement**: every rule-based splitter in this sub-chapter is deterministic (same input always produces the same chunks), which is what makes the incremental-ingestion manifest pattern, reproducible debugging, and consistent retrieval behavior all work correctly. Introducing any non-deterministic stage (agentic chunking, or any other LLM-in-the-loop ingestion step) is a decision with consequences well beyond chunk quality alone.
- **The full splitter landscape is really one recurring idea, applied to different structures**: prose → recursive character splitting; Q&A/Markdown/HTML → structure markers; code → language grammar; nested data → object graph traversal; irregular content → LLM judgment as a last resort. Every splitter covered across this entire sub-chapter is the same underlying question — "what is the most reliable signal available in this content for where a complete idea ends?" — answered differently per content type.

### Revision Summary

> Beyond the core general-purpose splitters, specialized variants exist for code (language-grammar-aware separators), HTML (header-tag structure), and nested JSON (object-graph traversal), each inheriting the same structure-reliability trade-off as Markdown/Q&A splitters. Agentic/LLM-based chunking removes rule-based logic entirely in favor of model judgment, at real cost and non-determinism — reserved for irregular content where every cheaper option has measurably failed, and notably incompatible with the determinism the incremental-ingestion manifest pattern assumes.

In [ ]:
"""
other_splitters.py
---------------------
Worked examples of three further splitter types: a code-aware recursive
splitter, an HTML header splitter, and a structure-aware JSON splitter.
Agentic/LLM-based chunking is described in theory only -- no code example,
since it's just "send text to an LLM with a chunking prompt and parse the
response," with no reusable mechanics beyond a prompt template.
"""

import re
import json
from document import make_document
from recursive_text_splitter import recursive_character_split  # reuse Topic 5's core logic

# ----------------------------------------------------------------------
# 1. CodeTextSplitter -- recursive splitting with language-aware separators
# ----------------------------------------------------------------------
PYTHON_SEPARATORS = [
    "\nclass ",      # class boundaries first -- highest-level structural unit
    "\ndef ",        # then function boundaries
    "\n\n",          # then blank-line-separated blocks
    "\n",            # then individual lines
    " ",              # then words
    "",               # absolute last resort: character cut
]


def code_text_split(code: str, chunk_size: int = 200, separators: list = None) -> list:
    """Same recursive mechanics as Topic 5, just with code-aware separators."""
    return recursive_character_split(code, chunk_size=chunk_size, separators=separators or PYTHON_SEPARATORS)


# ----------------------------------------------------------------------
# 2. HTMLHeaderTextSplitter -- splits on <h1>/<h2>/<h3>, mirrors the
#    Markdown header splitter from Topic 4 but for HTML tags.
# ----------------------------------------------------------------------
HTML_HEADER_RE = re.compile(r"<h([1-3])>(.*?)</h\1>", re.IGNORECASE)


def html_header_split(html: str) -> list:
    matches = list(HTML_HEADER_RE.finditer(html))
    if not matches:
        return [make_document(html.strip(), {"header_path": []})]

    documents = []
    header_stack = []

    for i, match in enumerate(matches):
        level = int(match.group(1))
        title = match.group(2).strip()

        header_stack = [h for h in header_stack if h[0] < level]
        header_stack.append((level, title))

        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(html)
        section_text = re.sub(r"<[^>]+>", "", html[start:end]).strip()  # strip remaining tags

        if section_text:
            documents.append(make_document(
                page_content=section_text,
                metadata={"header_path": [h[1] for h in header_stack]},
            ))

    return documents


# ----------------------------------------------------------------------
# 3. JSONSplitter -- structure-aware: walks the object graph, recursing
#    only into sub-structures that are still too large.
# ----------------------------------------------------------------------
def json_structure_split(data, chunk_size: int = 200, path: str = "$") -> list:
    serialized = json.dumps(data, ensure_ascii=False)

    if len(serialized) <= chunk_size:
        return [make_document(page_content=serialized, metadata={"json_path": path})]

    documents = []
    if isinstance(data, dict):
        for key, value in data.items():
            documents.extend(json_structure_split(value, chunk_size, path=f"{path}.{key}"))
    elif isinstance(data, list):
        for i, item in enumerate(data):
            documents.extend(json_structure_split(item, chunk_size, path=f"{path}[{i}]"))
    else:
        # a single scalar value that's somehow still oversized (e.g. a huge string)
        documents.append(make_document(page_content=serialized, metadata={"json_path": path}))

    return documents


if __name__ == "__main__":
    code_sample = (
        "class FDValidator:\n"
        "    def validate(self, fd_no):\n"
        "        return fd_no.startswith('BJ')\n\n"
        "    def normalize(self, fd_no):\n"
        "        return fd_no.strip().upper()\n"
    )
    print("--- CodeTextSplitter ---")
    for c in code_text_split(code_sample, chunk_size=60):
        print(f"  {c!r}")

    html_sample = (
        "<h1>FD Policy</h1><p>Overview text.</p>"
        "<h2>Premature Withdrawal</h2><p>A 1 percent penalty applies.</p>"
        "<h2>Senior Citizen Rate</h2><p>An additional 0.5 percent applies.</p>"
    )
    print("\n--- HTMLHeaderTextSplitter ---")
    for d in html_header_split(html_sample):
        print(f"  header_path={d['metadata']['header_path']}: {d['page_content']!r}")

    json_sample = {
        "document_code": "FD-PROD-02",
        "pages": [
            {"page_number": 1, "text": "Short page text."},
            {"page_number": 2, "text": "A much longer page of text that would exceed a small chunk size limit on its own and needs to be considered separately from page 1."},
        ],
    }
    print("\n--- JSONSplitter ---")
    for d in json_structure_split(json_sample, chunk_size=80):
        print(f"  json_path={d['metadata']['json_path']}: {d['page_content'][:60]!r}")